# MIE 402 — Pre-Lab 4: Spring–cart free response
**University of Massachusetts Amherst · Prof. Ehsan Roohi · Fall 2026**

## Submit to Canvas before YOUR Lab 4 section begins
Submit **one Word document or PDF** containing: (1) calculations with units; (2) mass/stiffness/damping predictions; (3) the analytical-versus-numerical, peak/log-fit, FFT and parameter-comparison figures generated below, each with a short caption; (4) answers to Questions 1–6 in your own words. **Do not submit a notebook or code instead of the document.**

**No programming is required.** Run the supplied cells from top to bottom; record and interpret outputs. Canvas controls the exact section-specific deadline.


## Why engineers use a ring-down test
After a bump, impact, or release, a suspension, machine mount, building floor, aircraft instrument, or packaged product vibrates freely. The frequency of that decay identifies the mass-stiffness combination. The rate of decay measures damping and settling. A frequency shift can reveal a stiffness or attachment change.

Lab 4 reproduces this system-identification problem with a Vernier cart, two springs, and eddy-current dampers.


## Model and variables
The cart displacement $x(t)$ is measured from equilibrium. After release, no external force acts:

$$m\ddot{x}+c\dot{x}+kx=0, \qquad k=k_L+k_R.$$

- $m$: total moving mass, including cart and every attached moving component [kg]
- $c$: viscous-equivalent damping coefficient [N s/m]
- $k$: equivalent stiffness of the left and right springs [N/m]
- $x_0=x(0)$ and $v_0=\dot{x}(0)$: initial displacement and velocity

For an underdamped response:

$$\omega_n=\sqrt{k/m},\quad f_n=\omega_n/(2\pi),\quad
\zeta=\frac{c}{2\sqrt{mk}},$$
$$\omega_d=\omega_n\sqrt{1-\zeta^2},\quad f_d=\omega_d/(2\pi),\quad
\alpha=\zeta\omega_n=\frac{c}{2m}.$$

The displacement has the form $x(t)=e^{-\alpha t}[A\cos(\omega_d t)+B\sin(\omega_d t)]$.


## How the model connects to measurements

The two pre-tensioned springs both create a restoring-force change opposite a small cart displacement: \(\Delta F_1=-k_1x\), \(\Delta F_2=-k_2x\); therefore \(k_{eq}=k_1+k_2\) [N/m]. From Newton's law, \(m\ddot x+c\dot x+k_{eq}x=0\). Here \(m\) [kg] includes all attachments, \(c\) [N·s/m] approximates viscous energy loss, \(x\) [m] is position relative to equilibrium, and \(t\) [s] is time.

Divide by \(m\): \(\ddot x+2\zeta\omega_n\dot x+\omega_n^2 x=0\), with \(\omega_n=\sqrt{k_{eq}/m}\) [rad/s], \(\zeta=c/(2m\omega_n)\) [dimensionless], \(\alpha=\zeta\omega_n=c/(2m)\) [1/s], and \(\omega_d=\sqrt{\omega_n^2-\alpha^2}\) [rad/s] for \(0<\zeta<1\). Thus \(T_d=2\pi/\omega_d\) [s] and \(f_d=1/T_d\) [Hz].

A displacement trace follows a decaying sinusoid bounded by \(\pm A e^{-\alpha t}\) [m]. For *same-sign* peaks \(p_i,p_{i+N}\), the logarithmic decrement **per cycle** is \(\delta=(1/N)\ln(p_i/p_{i+N})=\alpha T_d\), and \(\zeta=\delta/\sqrt{4\pi^2+\delta^2}\). A straight line fitted to \(\ln(p/(1\,\mathrm m))\) against peak time has slope \(-\alpha\), from which \(c=2m\alpha\). The FFT peak is a **frequency cross-check**, not a unique measurement of damping; finite duration, decay and bin spacing affect its location.

**Illustrative worked check (not experimental data):** If five complete cycles take 5.00 s and positive peaks decrease from 0.040 m to 0.020 m, \(T_d=1.00\) s, \(\delta=\ln(2)/5=0.1386\), \(\zeta\approx0.0221\), and \(\alpha=\delta/T_d=0.1386\) 1/s. For \(m=0.50\) kg, \(c=2m\alpha=0.1386\) N·s/m.

**Physical caveat:** The no-added-magnet configuration still has rolling, air and other residual losses. Eddy-current braking is only approximately linear in velocity over a suitable operating range. Hardware changes may alter moving mass as well as damping.

## New parameter set
Use the following values, which differ from the previous assignment:

| Quantity | Value |
|---|---:|
| total moving mass $m$ | 1.35 kg |
| damping coefficient $c$ | 0.72 N s/m |
| left spring $k_L$ | 34 N/m |
| right spring $k_R$ | 46 N/m |
| initial displacement $x_0$ | 0.080 m |
| initial velocity $v_0$ | 0.040 m/s |

### Question 1 - calculations (6 points)
Calculate $k$, $\omega_n$, $f_n$, $\zeta$, $\omega_d$, $f_d$, and $\alpha$. Show units and at least three significant digits.


In [ ]:
# Provided setup: run this cell
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks

m = 1.35       # kg
c = 0.72       # N s/m
k_left = 34.0  # N/m
k_right = 46.0 # N/m
x0 = 0.080     # m
v0 = 0.040     # m/s
k = k_left + k_right

wn = np.sqrt(k/m)
fn = wn/(2*np.pi)
zeta = c/(2*np.sqrt(m*k))
wd = wn*np.sqrt(1-zeta**2)
fd = wd/(2*np.pi)
alpha = zeta*wn

print(f"k = {k:.3f} N/m")
print(f"wn = {wn:.4f} rad/s, fn = {fn:.4f} Hz")
print(f"zeta = {zeta:.5f}")
print(f"wd = {wd:.4f} rad/s, fd = {fd:.4f} Hz")
print(f"alpha = {alpha:.4f} 1/s")


### Question 2 - predictions before plotting (3 points)
For each change, state what happens to oscillation frequency and decay rate, and explain why.

1. Double $m$ while keeping $k$ and $c$ fixed.
2. Double both spring stiffnesses while keeping $m$ and $c$ fixed.
3. Increase the number of magnetic dampers while keeping the same cart and springs.

Write your predictions before running the next cells.


## Analytical and numerical responses
The first curve uses the closed-form underdamped solution. The second curve integrates the same differential equation numerically. They should overlap if the equations and initial conditions are consistent.


In [ ]:
# Provided analytical solution: run this cell
t = np.linspace(0, 12, 2401)
A = x0
B = (v0 + alpha*x0)/wd
x_analytical = np.exp(-alpha*t)*(A*np.cos(wd*t) + B*np.sin(wd*t))

def rhs(t, y):
    x, v = y
    return [v, -(c/m)*v - (k/m)*x]

sol = solve_ivp(rhs, [t[0], t[-1]], [x0, v0], t_eval=t,
                rtol=1e-10, atol=1e-12)
x_numerical = sol.y[0]

fig, ax = plt.subplots(figsize=(10,4.8))
ax.plot(t, 1000*x_analytical, lw=2.2, label='analytical')
ax.plot(t, 1000*x_numerical, '--', lw=1.5, label='numerical ODE')
ax.set(xlabel='Time (s)', ylabel='Displacement (mm)',
       title='Analytical and numerical free response')
ax.grid(True, alpha=.3); ax.legend(); plt.show()

print('Maximum absolute difference:',
      np.max(np.abs(x_analytical-x_numerical)), 'm')


### Question 3 - model comparison (2 points)
Include the analytical/numerical figure. Do the curves match? Explain what agreement tells you, and name one real laboratory effect that neither ideal curve may represent.


## Estimate frequency and damping from the response
The code removes the equilibrium offset, finds positive peaks, fits a line to $\ln|x_{peak}|$, and compares the measured decay rate with $\alpha=\zeta\omega_n$. The slope should be approximately $-\alpha$.


In [ ]:
# Provided peak and logarithmic-decay analysis: run this cell
x = x_numerical.copy()  # ideal model is already centered at x_eq = 0
min_spacing = max(1, int(0.65*(1/fd)/(t[1]-t[0])))
peak_idx, _ = find_peaks(x, distance=min_spacing,
                         prominence=0.002)
tp = t[peak_idx]
xp = x[peak_idx]
valid = xp > 0
tp, xp = tp[valid], xp[valid]

coef = np.polyfit(tp, np.log(xp), 1)
alpha_fit = -coef[0]
omega_d_fit = 2*np.pi/np.mean(np.diff(tp))
omega_n_fit = np.sqrt(omega_d_fit**2 + alpha_fit**2)
zeta_fit = alpha_fit/omega_n_fit
fit_log = np.polyval(coef, tp)
r2 = 1 - np.sum((np.log(xp)-fit_log)**2)/np.sum((np.log(xp)-np.mean(np.log(xp)))**2)

fig, axs = plt.subplots(1,2,figsize=(12,4.4))
axs[0].plot(t, 1000*x, label='response')
axs[0].plot(tp, 1000*xp, 'o', label='selected positive peaks')
axs[0].plot(t, 1000*np.exp(coef[1]-alpha_fit*t), '--', label='fitted envelope')
axs[0].set(xlabel='Time (s)', ylabel='Displacement (mm)', title='Selected peaks and envelope')
axs[0].grid(True,alpha=.3); axs[0].legend()
axs[1].plot(tp, np.log(xp), 'o', label='ln(peak amplitude)')
axs[1].plot(tp, fit_log, '-', label='linear fit')
axs[1].set(xlabel='Peak time (s)', ylabel='ln(peak displacement / 1 m)', title='Log-peak regression')
axs[1].grid(True,alpha=.3); axs[1].legend()
plt.tight_layout(); plt.show()

print(f"input alpha = {alpha:.5f} 1/s")
print(f"fitted alpha = {alpha_fit:.5f} 1/s")
print(f"fitted zeta = {zeta_fit:.5f}")
print(f"log-fit R^2 = {r2:.6f}")


### Question 4 - damping from decay (3 points)
Include both peak-analysis panels. Report the fitted $\alpha$, fitted $\zeta$, and $R^2$. Compare fitted and input values using percent difference. Why must peaks of the same sign be used consistently?


## Frequency spectrum
The FFT supplies a frequency cross-check. A finite decaying record produces a broadened peak near $f_d$. It does not determine damping by itself. Displacement spectrum remains in displacement units; there is no degree conversion.


In [ ]:
# Provided one-sided FFT: run this cell
dt = t[1]-t[0]
N = len(x)
freq = np.fft.rfftfreq(N, dt)
amp = 2*np.abs(np.fft.rfft(x))/N
amp[0] *= 0.5
peak_band = (freq > 0.1) & (freq < 5.0)
f_fft = freq[peak_band][np.argmax(amp[peak_band])]

fig, ax = plt.subplots(figsize=(9,4.6))
ax.plot(freq, amp, lw=1.8)
ax.axvline(f_fft, color='crimson', ls='--', label=f'FFT peak = {f_fft:.3f} Hz')
ax.axvline(fd, color='black', ls=':', label=f'theory fd = {fd:.3f} Hz')
ax.set_xlim(0, 3)
ax.set(xlabel='Frequency (Hz)', ylabel='One-sided displacement amplitude (m)',
       title='Frequency spectrum of the decaying response')
ax.grid(True,alpha=.3); ax.legend(); plt.show()
print(f"FFT peak = {f_fft:.4f} Hz; theory fd = {fd:.4f} Hz")


### Question 5 - FFT interpretation (2 points)
Include the spectrum. Compare its peak with the theoretical $f_d$. Explain why the peak has a finite width and why the FFT cannot replace the decay-envelope fit for estimating damping.


## Parameter-change comparison


In [ ]:
# Provided comparison code: run this cell
cases = {
    'base': (m, c, k),
    'double mass': (2*m, c, k),
    'double stiffness': (m, c, 2*k),
    'double damping': (m, 2*c, k),
}
fig, ax = plt.subplots(figsize=(10,5))
for name,(mi,ci,ki) in cases.items():
    wni=np.sqrt(ki/mi); zi=ci/(2*np.sqrt(mi*ki)); wdi=wni*np.sqrt(max(0,1-zi**2)); ai=ci/(2*mi)
    Bi=(v0+ai*x0)/wdi
    xi=np.exp(-ai*t)*(x0*np.cos(wdi*t)+Bi*np.sin(wdi*t))
    ax.plot(t,1000*xi,label=f'{name}: fd={wdi/(2*np.pi):.2f} Hz, zeta={zi:.3f}')
ax.set(xlim=(0,6),xlabel='Time (s)',ylabel='Displacement (mm)',title='Effect of changing one parameter')
ax.grid(True,alpha=.3);ax.legend();plt.show()


### Question 6 - explain the comparison (4 points)
Include the comparison figure. For each changed case, describe separately:

- how the time between peaks changes,
- how the decay envelope changes,
- which formula explains the change.

Finally, identify one reason the no-magnet cart response might not have a perfectly exponential envelope.


## Before arriving at Lab 4
Bring or have access to this completed submission. During the experiment, record the **total moving mass**, spring pair, number of dampers, initial displacement, sample rate, record duration, and CSV file name for every case. Preserve every raw CSV file for the combined Labs 4-5 report.


## Final interpretation checkpoint

Before submission, explain why a 49%-larger FFT peak **does not necessarily mean** less damping. Identify at least two factors among initial displacement, duration, windowing, units, and damping. Explain what patterned residuals would suggest about the linear viscous model, and why adding magnetic hardware requires reweighing the moving system.

In the real laboratory, distinguish the supplied *illustrative* notebook signals from Vernier CSV measurements: record each actual spring ID, mass, magnet count, sampling rate, initial displacement and filename.